<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Apêndice F: Abordagens comuns para avaliação de LLMs

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.0
torch version: 2.7.1
tokenizers version: 0.21.2


&nbsp;
## F.1 Entendendo os principais métodos de avaliação de LLMs

- Sem código nesta seção

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 Avaliando a acurácia de escolha de resposta

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px">

- Note que esta figura mostra uma versão simplificada de uma avaliação baseada em múltipla escolha (como o MMLU), em que comparamos a letra gerada na saída com a letra da resposta correta
- Na prática, variantes disso incluem a pontuação por log-probability, em que, em vez de checar apenas a letra final, calculamos o quão provável o modelo considera cada resposta candidata
- Para modelos de raciocínio, isso também pode envolver avaliar a probabilidade de a resposta correta ser produzida quando alimentada no modelo
- Em qualquer um dos casos, a avaliação ainda checa se o modelo seleciona uma das respostas predefinidas
- (Os scores de probabilidade de saída são discutidos em mais detalhe no capítulo 4, onde melhoramos a função de geração de texto)

&nbsp;
#### F.2.1 Carregando o modelo

In [2]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
torch.set_float32_matmul_precision("high")

# If you have compatibility issues, try to
# uncomment the line below and rerun the notebook
# device = "cpu"

WHICH_MODEL = "base"

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")


model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)


USE_COMPILE = False  # Set to true to enable compilation
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True
  model = torch.compile(model)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
✓ qwen3/tokenizer-base.json already up-to-date


&nbsp;
#### F.2.2 Verificando a letra de resposta gerada

In [3]:
example = {
    "question": (
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],
    "answer": "D",
}

def format_prompt(example):
    return (
        f"{example['question']}\n"
        f"A. {example['choices'][0]}\n"
        f"B. {example['choices'][1]}\n"
        f"C. {example['choices'][2]}\n"
        f"D. {example['choices'][3]}\n"
        "Answer: "  # trailing space encourages a single-letter next token
    )

prompt = format_prompt(example)
print(prompt)

How many ways are there to put 4 distinguishable balls into 2 indistinguishable boxes?
A. 7
B. 11
C. 16
D. 8
Answer: 


---


- Você pode carregar exemplos do dataset MMLU diretamente pela biblioteca `datasets` (que pode ser instalada via `pip install datasets` ou `uv add datasets`):

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# Inspect the first example from test set:
example = dataset["test"][0]
print(example)
```

- Acima, usamos o subconjunto `"high_school_mathematics"`; para obter uma lista dos outros subconjuntos, use o seguinte código:


```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---

In [4]:
prompt_ids = tokenizer.encode(prompt)
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)

- Geramos alguns tokens e extraímos a primeira ocorrência da letra A/B/C/D que o modelo imprime:

In [5]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def predict_choice(
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None
    for t in generate_text_basic_stream_cache(
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())
        for letter in answer:
            letter = letter.upper()
            if letter in "ABCD":
                pred = letter
                break
        if pred:  # stop as soon as a letter appears
            break
    return pred

In [6]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)

print(
    f"Generated letter: {pred1}\n"
    f"Correct? {pred1 == example['answer']}"
)

Generated letter: C
Correct? False


&nbsp;
### F.3 Usando verificadores para checar respostas

- Sem código nesta seção (veja o capítulo 3)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 Comparando modelos usando preferências e leaderboards

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px">

- Rating Elo ("algoritmo do 400"), inspirado nos rankings de xadrez: https://en.wikipedia.org/wiki/Performance_rating_(chess)
- Note que o LM Arena migrou para um modelo estatístico de Bradley-Terry que fornece scores em uma escala parecida com a do Elo; no entanto, o mesmo conceito de ranqueamento par a par continua valendo

In [7]:
# Pairwise "arena votes" where the first model is the winner and
# the second model is the loser
votes = [
    ("GPT-5", "Claude-3"),  # First match-up: GPT-5 was preferred over Claude-3
    ("GPT-5", "Llama-4"),
    ("Claude-3", "Llama-3"),
    ("Llama-4", "Llama-3"),
    ("Claude-3", "Llama-3"),
    ("GPT-5", "Llama-3"),
]

In [8]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # Initialize all models with the same base rating
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # Update ratings after each match
    for winner, loser in vote_pairs:

        # Expected score for the current winner given the ratings
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor determines sensitivity of rating updates
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings

In [9]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)

for model in sorted(ratings, key=ratings.get, reverse=True):
    print(f"{model:8s} : {ratings[model]:.1f}")

GPT-5    : 1043.7
Claude-3 : 1015.2
Llama-4  : 1000.7
Llama-3  : 940.4


- O score esperado do vencedor é calculado da seguinte forma:

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- Intuição:
    - Se rating_winner >> rating_loser:
       - expoente → muito negativo
       - denominador ≈ 1
       - expected_winner ≈ 1 (vitória quase certa)
    - Se rating_winner << rating_loser:
       - expoente → muito positivo
       - denominador → muito grande
       - expected_winner ≈ 0 (derrota quase certa)
    - Se rating_winner == rating_loser:
       - expoente = 0
       - denominador = 2
       - expected_winner = 0.5 (partida equilibrada)

&nbsp;
### F.5 Julgando respostas com outros LLMs

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- Nesta seção, automatizamos a avaliação de respostas do LLM ajustado usando outro LLM, maior
- Em particular, usamos um modelo gpt-oss de 20 bilhões de parâmetros, ajustado para instruções, da Open AI, que pode ser rodado localmente via ollama ([https://ollama.com](https://ollama.com))

- O ollama é uma aplicação de código aberto para rodar LLMs de forma eficiente
- É um wrapper em torno do llama.cpp ([https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)), que implementa LLMs em C/C++ puro para maximizar a eficiência
- Note que é uma ferramenta para usar LLMs para gerar texto (inferência), não para treinar ou fazer fine-tuning de LLMs
- Antes de rodar o código abaixo, instale o ollama visitando [https://ollama.com](https://ollama.com) e seguindo as instruções (por exemplo, clicando no botão "Download" e baixando a aplicação ollama para o seu sistema operacional)

- Usuários de macOS e Windows: clique na aplicação ollama que você baixou; se ela perguntar se deseja instalar o uso por linha de comando, responda "yes"
- Usuários de Linux podem usar o comando de instalação fornecido no site do ollama
- Há 3 formas de rodar o ollama no nosso computador:

**1. `ollama serve`**

- Isso roda o backend do ollama como um servidor, normalmente em `http://localhost:11434`. Ele não carrega um modelo até que o chamemos pela API. É isso que queremos se formos usar o ollama pelo Python.

**2. `ollama run gpt-oss:20b`**

- Este é um wrapper de conveniência. Se o servidor ainda não estiver rodando, ele o inicia, então baixa o modelo (na primeira vez) e nos deixa em um terminal interativo onde podemos conversar com o modelo. Nos bastidores, ele usa a mesma API do servidor.

**3. Aplicativo desktop do Ollama**

- Isso roda o mesmo backend automaticamente e fornece uma interface gráfica por cima dele (como mostrado na figura acima).
Ele também aplica valores padrão (system prompt, temperature, stop sequences), o que pode explicar por que as respostas parecem diferentes do uso direto da API.

---

**Nota**:

- Ao rodar `ollama serve` no terminal, como descrito acima, você pode encontrar uma mensagem de erro dizendo `Error: listen tcp 127.0.0.1:11434: bind: address already in use`
- Se for o caso, tente usar o comando `OLLAMA_HOST=127.0.0.1:11435 ollama serve` (e, se esse endereço também estiver em uso, tente incrementar os números de um em um até encontrar um endereço livre)

---

- Por exemplo, para experimentar o ollama, podemos usar `ollama run gpt-oss:20b` para testar o modelo gpt-oss 20B, de 20 bilhões de parâmetros. O modelo
  (cerca de 13 GB) será baixado automaticamente na primeira vez que você rodar
  este comando. (Como alternativa, você pode usá-lo no aplicativo desktop, de forma parecida com a figura anterior.)

```bash
ollama run gpt-oss:20b
```


- A saída fica assim:

```
$ ollama run gpt-oss:20b
pulling manifest
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B
verifying sha256 digest
writing manifest
removing unused layers
success
```

- Para mais informações sobre o gpt-oss, veja meu artigo aprofundado, [From GPT-2 to gpt-oss: Analyzing the Architectural Advances](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the)
- Usar o ollama com o modelo `"gpt-oss:20b"` (um modelo de 20B parâmetros) exige 13 GB de RAM; se sua máquina não suportar isso, você pode tentar um modelo menor, como o `qwen3:4b`, de 4B parâmetros, que exige apenas cerca de 4 GB de RAM
- Como alternativa, você também pode usar o gpt-oss maior, de 120 bilhões (`qwen3:235b`), ou até o modelo Qwen3 de 235 bilhões de parâmetros (`qwen3:235b`), se sua máquina suportar
- Depois que o download terminar, você verá um prompt de linha de comando que permite conversar com o modelo
- Experimente um prompt como "What is 1+2?", que deve retornar uma saída parecida com a seguinte

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```

- Você pode encerrar essa sessão usando a entrada `/bye`

- O código a seguir verifica se a sessão do ollama está rodando corretamente antes de prosseguirmos para usar o ollama na avaliação das respostas do conjunto de teste que geramos na seção anterior

In [10]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


- Agora, uma alternativa ao comando `ollama run` que usamos antes para interagir com o modelo é via a REST API dele, em Python, através da seguinte função
- Antes de rodar as próximas células deste notebook, certifique-se de que o ollama ainda está rodando (as células de código anteriores devem imprimir `"Ollama running: True"`)
- Em seguida, rode a célula de código a seguir para consultar o modelo

In [11]:
import json
import requests


def query_model(
    prompt,
    model="gpt-oss:20b",
    # If you used OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # update the address from 11434 to 11435
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Send the POST request
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

In [12]:
ollama_model = "gpt-oss:20b"
result = query_model("What is 1+2?", ollama_model)
print(result)

3


- Agora, usando a função `query_model` que definimos acima, podemos avaliar as respostas do nosso próprio modelo

In [16]:
def rubric_prompt(instruction, reference_answer, model_answer):
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt

- O `model_answer` poderia ser a resposta produzida pelo nosso próprio modelo; aqui, deixamos uma possível resposta do modelo fixa no código, por simplicidade

In [17]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)

You are a fair judge assistant. You will be given an instruction, a reference answer, and a candidate answer to evaluate, according to the following rubric:

1: The response fails to address the instruction, providing irrelevant, incorrect, or excessively verbose content.
2: The response partially addresses the instruction but contains major errors, omissions, or irrelevant details.
3: The response addresses the instruction to some degree but is incomplete, partially correct, or unclear in places.
4: The response mostly adheres to the instruction, with only minor errors, omissions, or lack of clarity.
5: The response fully adheres to the instruction, providing a clear, accurate, and relevant answer in a concise and efficient manner.

Now here is the instruction, the reference answer, and the response.

Instruction:
If all birds can fly, and a penguin is a bird, can a penguin fly?

Reference Answer:
Yes, according to the premise that all birds can fly, a penguin can fly.

Answer:
Yes – 

In [18]:
result = query_model(rendered_prompt, ollama_model)
print(result)

**Score: 5**

The candidate answer directly addresses the question, correctly applies the given premises, and concisely states that a penguin would be able to fly. It is accurate, relevant, and clear.
